In [ ]:
# !pip install openai langchain-openai

## Custom Model Assemlbly to Deployment

create an instance of the datarobot Client

For MTS, your `endpoint = "https://app.datarobot.com/api/v2"`.  You can get your DataRobot Api Token by going to the UI -> Clicking the avatar in the top right -> Developer Tools -> Crate New Key or use an existing one

In [1]:
import datarobot as dr 
import os 
client = dr.Client()

## Grab an appropriate environment

In [2]:
dr.ExecutionEnvironment.list("genai")

[ExecutionEnvironment('[DataRobot] Python 3.11 GenAI Agents'),
 ExecutionEnvironment('[GenAI][NVIDIA] Generic image used to proxy requests to NIM'),
 ExecutionEnvironment('[GenAI] Python 3.12 with Moderations'),
 ExecutionEnvironment('[GenAI][NVIDIA] NeMo Retriever Text Embedding NIM'),
 ExecutionEnvironment('[PREVIEW][GenAI][NVIDIA] NIM Qwen-2.5-7b-instruct'),
 ExecutionEnvironment('[GenAI][NVIDIA] NIM Llama-3.1-70b-instruct'),
 ExecutionEnvironment('[GenAI] vLLM Inference Server'),
 ExecutionEnvironment('[DataRobot][NVIDIA] Python 3.11 GenAI'),
 ExecutionEnvironment('[GenAI][NVIDIA] NIM Llama-3.1-8b-instruct'),
 ExecutionEnvironment('[GenAI] Python 3.11 with Moderations'),
 ExecutionEnvironment('[DataRobot] Python 3.11 GenAI'),
 ExecutionEnvironment('[DEPRECATED] Python 3.9 GenAI')]

In [3]:
environment = dr.ExecutionEnvironment.list("Python 3.11 GenAI Agents").pop()
environment

ExecutionEnvironment('[DataRobot] Python 3.11 GenAI Agents')

## Create an entry for the model in the Custom Model Workshop

In [4]:
## for regression all assets provided in python-version folder correspond to a regression problem. 
cm = dr.CustomInferenceModel.create(
    "Proxied LLM", 
    target_name="resultText",
    target_type= dr.enums.TARGET_TYPE.TEXT_GENERATION)

## create crenentials in app (or sdk) 

check out the [docs](https://docs.datarobot.com/en/docs/platform/acct-settings/stored-creds.html#credentials-management)

We'll need a credential for an llm and datarobot (api tokens)

## Add a version to the entry 

The path `./src` contains all artifacts datarobot would need to ensure it can score with the model.  see more details [here](https://github.com/datarobot/datarobot-user-models?tab=readme-ov-file#custom-inference-models-reference-)

In [5]:
LLM_CREDENTIAL = [cred for cred in dr.Credential.list() if cred.name == "DATAROBOT_API_TOKEN"].pop()

In [6]:

runtime_parameter_values = [
    dr.models.runtime_parameters.RuntimeParameterValue(field_name = "MODEL", value = "anthropic/claude-sonnet-4-6", type = "string"),
    dr.models.runtime_parameters.RuntimeParameterValue(field_name = "API_BASE_URL", value = "https://app.datarobot.com/api/v2/genai/llmgw", type = "string"),
    dr.models.runtime_parameters.RuntimeParameterValue(field_name = "API_TOKEN", value = LLM_CREDENTIAL.credential_id, type = "credential"),
]



In [7]:
cmv = dr.CustomModelVersion.create_clean(cm.id, 
                                        base_environment_id = environment.id,
                                        folder_path = "./src", 
                                        runtime_parameter_values = runtime_parameter_values,   
                                        )

## or 
# cmv = dr.CustomModelVersion.create_from_previous(cm.id, base_environment_id = environment.id, runtime_parameter_values = runtime_parameter_values)

## Build the custom model environment if necessary

this would be required if you added a  `requirements.txt`


In [8]:
try:
    build = dr.CustomModelVersionDependencyBuild.start_build(cm.id, cmv.id, max_wait = 1200)
    build.build_status
except Exception as e:
    print(e)

422 client error: {'message': 'Version 69df988f100854bef772dac4 has no additional dependencies'}


## Register the Model

In [9]:
## register the custom model version in the dr model registry 
registered_model_version = dr.RegisteredModelVersion.create_for_custom_model_version(
    custom_model_version_id =  cmv.id, 
    name = "LLM Proxy", 
    registered_model_name = "LLM Proxy",
    description = "LLM Proxy"
)

In [10]:
registered_model_version.build_status

Wait for the model package to be built

In [11]:
import time 
buildStatus = "inProgress"
while buildStatus == "inProgress":
    buildStatus = client.get(f"registeredModels/{registered_model_version.registered_model_id}/versions/{registered_model_version.id}").json()["buildStatus"]
    time.sleep(5)

## Deploy the model

In [12]:
## this will grab the first serverless prediction environment returned
prediction_environment = [ pe for pe in dr.PredictionEnvironment.list() if pe.platform == "datarobotServerless"].pop()

In [13]:
deployment = dr.Deployment.create_from_registered_model_version(
    registered_model_version.id,
    prediction_environment_id=prediction_environment.id,
    label = "LLM Proxy",
)

## Test Deployment

In [ ]:
import requests
import json
import os 

url = f"{client.endpoint}/deployments/{deployment.id}/chat/completions"
messages = [
    {
      "role": "user",
      "content": "tell me a joke",
    }
  ]
headers = {
  'Authorization': f'Bearer {os.environ.get("DATAROBOT_API_TOKEN")}',
  'Content-Type': 'application/json'
}

response = requests.request("POST", url, headers=headers, json=dict(messages = messages, model = "proxied-llm", stream = False))

response.json()


{'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'annotations': None,
    'audio': None,
    'content': "Here's one:\n\nWhy don't scientists trust atoms?\n\n**Because they make up everything!** 😄\n\nWant to hear another one?",
    'function_call': None,
    'provider_specific_fields': {'citations': None, 'thinking_blocks': None},
    'refusal': None,
    'role': 'assistant',
    'tool_calls': None}}],
 'created': 1776261496,
 'datarobot_association_id': 'ff6919b9-640c-455a-a5d1-a4dabcae8d0a',
 'id': 'chatcmpl-2b2e3e5b-cd2e-4932-a09a-29d62c351646',
 'model': 'claude-sonnet-4-6',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'cache_creation_input_tokens': 0,
  'cache_read_input_tokens': 0,
  'completion_tokens': 34,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': None,
   'text_tokens': 34},
  'infe

## OPENAI

In [15]:
import os
from openai import OpenAI
url = f"{client.endpoint}/deployments/{deployment.id}"
client = OpenAI(
    base_url=url,
    api_key=os.environ["DATAROBOT_API_TOKEN"],
)

r = client.chat.completions.create(
    model="proxied-llm",
    messages=[{"role": "user", "content": "tell me a joke"}],
)

print(r.choices[0].message.content)

Here's one:

Why don't scientists trust atoms?

**Because they make up everything!** 😄

Want to hear another one?


In [16]:
stream = client.chat.completions.create(
    model="proxied-llm",
    messages=[{"role": "user", "content": "tell me a joke"}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)
print()

Here's one:

Why don't scientists trust atoms?

**Because they make up everything!** 😄

Want to hear another one?


In [18]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


llm = ChatOpenAI(
    base_url= url,
    api_key=os.environ["DATAROBOT_API_TOKEN"],
    model="proxied-llm",
)


In [19]:
# Non-streaming
print(llm.invoke([HumanMessage(content="Say hi in one sentence.")]).content)



Hi there! Hope you're having a wonderful day! 😊


In [20]:
# Streaming
for chunk in llm.stream([HumanMessage(content="tell me a joke")]):
    print(chunk.content, end="", flush=True)
print()


Here's one:

Why don't scientists trust atoms?

**Because they make up everything!** 😄

Want to hear another one?
